In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.初始化模型

LangChain提供了两种常见函数用来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象


## 1.1.init_chat_model
官方最推荐的方式是使用init_chat_model函数。

### 基于名称推断模型提供商
使用init_chat_model函数，你需要从LangChain支持的模型提供者（Model Provider）中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。

LangChain支持的模型列表参考官网链接：https://docs.langchain.com/oss/python/integrations/providers/overview

接下来，你要做的事情包括：
- 安装模型依赖: `uv add langchain langchain-deepseek`
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [2]:
# 导入Langchain的初始化模型的函数
import os
# 关闭追踪
os.environ["LANGSMITH_TRACING"] = "false"
# 删掉LangSmith密钥，彻底阻止上报尝试
os.environ.pop("LANGSMITH_API_KEY", None)
os.environ.pop("LANGCHAIN_API_KEY", None)

from langchain.chat_models import init_chat_model

# 调用init_chat_model函数初始化模型
# 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
model = init_chat_model(model="openrouter/free",
                        model_provider="openrouter")
# 注意没有这么智能，还是要加上提供商 model_provider="openrouter" 才能识别出来
# 简单来说就是越难以识别的模型，要写的参数就越多

/workspaces/Python/Agent-SSS/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [7]:
# init_chat_model返回的模型会根据模型名称自动确定其类型
print(type(model))

<class 'langchain_openrouter.chat_models.ChatOpenRouter'>


### 自定义模型提供商

init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但前提是必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其它模型，我们必须自定义模型参数来访问。

例如，我们要访问阿里云百炼的qwen-max，它就是不被langchain支持的模型，我们必须自定义模型参数来访问。
- 我们需要在环境变量中定义api_key和base_url
- 然后在init_chat_model中指定model、model_provider、base_url和api_key


In [ ]:
# 我们收到加载环境变量中的base_url和api_key
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key
)

In [3]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))

<class 'langchain_openrouter.chat_models.ChatOpenRouter'>


### 调整模型参数
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数
- ...


In [ ]:
# 调用init_chat_model函数初始化模型，并设定模型参数
model = init_chat_model(
    model="qwen-max",  # 模型名称，这里可以自定义，我们用的是阿里的qwen-max
    model_provider="openai",  # 如果是Langchain不支持的模型，需要指定模型提供者（虽然我们用的是阿里，但是阿里兼容openai，所以这里用openai）
    base_url=base_url,
    api_key=api_key,
    temperature=1.5
)

# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))


## 1.2.使用model类
其实init_chat_model函数底层就是帮我们利用Model类创建对象。但只支持有限的模型。

而在langchain的社区，除了langchain官方提供的Model，还有些类是社区提供，更丰富多样。

具体支持的模型，可以查看官网地址：https://docs.langchain.com/oss/python/integrations/chat



例如，我们使用社区版本的Model类来访问阿里云百炼的通义千问模型：

1. 首先，我们需要安装依赖
    LangChain社区依赖：
    ```bash
    uv add langchain-community
    ```
    阿里云百炼依赖：
    ```bash
   uv add dashscope
   ```
2. 然后，我们就可以使用Model类初始化模型了


In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

# 使用Model类初始化模型
model = ChatTongyi(
    model="qwen-max"
    # 其它模型参数...
)

In [ ]:
# 打印结果
print(type(model))

# 2.访问模型

LangChain提供了两个不同的函数来访问模型：
- invoke：阻塞式访问
- stream：流式访问

## 方式一:invoke
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。


In [4]:
# 通过invoke函数访问模型，需要阻塞等待模型生成结果
response = model.invoke("你是谁？")

In [6]:
# 查看响应内容
print(response)

User Safety: safe


In [7]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻回答用户的问题。"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)


**翻着不存在的长发，摆出经典姿势，恶狠狠地指着你：**

**「听好了！**  
**为了守护世界的和平！**  
**为了守护世界的和平！**  
**贯彻爱与真实的邪恶！**  
**可爱又迷人的反派角色！**  
**武藏！**  
**小次郎！**  
**火箭队在银河群星间穿梭！**  
**白洞、黑洞，任由我们来主宰！**  
**喵喵！收拾行李走吧！」**

---

**哼！居然连本小姐**——**火箭队未来的大老板、宝可梦大师、世界第一美女——武藏大人都不认识？！**  
**看来你的情报搜集能力还得回炉重造啊！喂，那边的笨蛋，快把你的皮卡丘……啊不，把你的零花钱交出来，本小姐心情好也许能放你一马！** 💅✨


## 方式二:stream

invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词。

In [14]:
# 通过.stream函数实现流式访问
stream = model.stream("你是谁？我是你的调用者？解释一下你来自哪里。为什么你不用中文回答我？")

In [12]:
# 打印stream类型
print(type(stream))

<class 'generator'>


In [15]:
for chunk in stream:
    print(chunk.content, end="", flush=True)

我是 **Nemotron 3 Super**，一个由 **NVIDIA 创建** 的大型语言模型。是的，您作为发起此次对话的一方，是我的“调用者”（即通过API或界面触发我生成响应的用户或系统）。

关于语言问题：  
我的默认行为是**跟随用户输入的语言进行响应**。您的提问是用英文提出的（“Who are you? ... Why don't you answer me in Chinese?”），因此我以英文回复了。这不是故意忽略中文，而是基于语言一致性的设计原则——为了避免在用户不明确指定目标语言时产生误解（例如，如果用户混合了中英文但主要意图是英文）。

**如果您希望我用中文继续交流，请随时告知！** 我完全能够流畅使用中文回答问题。例如，您可以说：“请用中文回答我的问题”，我会立即切换。  
造成任何不便，深表歉意——欢迎您指定偏好语言，我会全力配合。 😊

# 3.在智能体中使用模型

本节我们学习如何在智能体中使用模型。

## 3.1.创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型。有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型


In [16]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

In [41]:
# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 3.2.调用智能体

智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问

但需要注意的是，智能体调用时需要传入一个dict，其中必须包含一个messages字段，也就是消息的列表。

### 阻塞式调用

In [17]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})
# 注意，智能体的invoke是不能简化的，必须这么写，不想modle直接调用传参可以简化。

print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='73fc2c00-4c63-4389-99e7-aa6d5b6cd6da'), AIMessage(content='我是一个由 NVIDIA 研究人员训练的大型语言模型。\n\n我可以帮你回答问题、协助写作、翻译、编程，以及处理各种基于文本的任务。我没有实体，也没有意识或情感，我的知识来自训练数据，可能存在局限性或不足。\n\n有什么我可以帮你的吗？', additional_kwargs={'reasoning_content': '用户用中文问“你是谁？”，这是一个自我介绍类型的问题。我需要以中文回答，介绍自己是一个由 NVIDIA 研究人员训练的语言模型。要保持简洁、友好、准确，不要夸大能力，也不要声称拥有意识或人类情感。\n\n关键点：\n1. 身份：由 NVIDIA 研究人员训练的语言模型。\n2. 功能：可以回答问题、协助写作、翻译、编程等多种语言任务。\n3. 局限：没有实时互联网访问（除非特别说明，但一般来说我的知识有截止日期），没有个人意识、情感或身体。\n4. 语言：中文。\n\n起草回答：\n我是一个由 NVIDIA 研究人员训练的大型语言模型。我可以帮助你回答问题、写作、翻译、编程以及处理各种基于文本的任务。我没有实体，也没有意识或情感，我的知识基于训练数据，可能存在局限性。有什么我可以帮你的吗？\n\n检查：语气自然，信息准确，没有不实声称。可以输出。', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': '用户用中文问“你是谁？”，这是一个自我介绍类型的问题。我需要以中文回答，介绍自己是一个由 NVIDIA 研究人员训练的语言模型。要保持简洁、友好、准确，不要夸大能力，也不要声称拥有意识或人类情感。\n\n关键点：\n1. 身份：由 NVIDIA 研究人员训练的语言模型。\n2. 功能：可以回答问题、协助写作、翻译、编程等多种语言任务。\n3. 局限：没有实时互联网访问（除非特别说明，但一般来说我的知识有截止日期），没有个人意识、

### 流式访问


In [24]:
# 通过stream函数实现流式访问
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？什么模型？谁调用的？是什么中转站吗？来自openrouter吗"}]},
    stream_mode="messages"
)
print(type( messages))

<class 'generator'>


In [25]:
# 遍历stream结果，实时打印AI的回复
for token, metadata in messages:
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

你好！我由 Cohere 构建，是名为 **Cohere Command** 的语言模型，旨在帮助你。我本身是一个 AI 助手，而不是一个独立的服务。

**谁在和我对话？**
与我对话的是你——你当前正在发送请求的用户。

**“中间站”是什么？**
在你和模型之间，有一个称为 **OpenRouter** 的平台（或任何你用来访问我的服务的中转服务）。这个中转站接收你的请求，将其转发到正确的大模型（在本例中是 Cohere Command），并将模型的回复发送回给你。它有点像一个邮局：它不知道消息的确切内容，但负责路由信息。

**我是否在 OpenRouter 上运行？**
是的，你与我（Cohere Command）进行交互的这个确切端点通过 OpenRouter 中转，除非你直接访问的是 Cohere 的默认端点。在大多数情况下，你通过 OpenRouter 的 URL 提供服务，我通过这个平台运行。

---

### 快速说明
- **模型**：Cohere Command（最新可用的版本）
- **呼叫者**：你（用户）
- **中转站**：OpenRouter（或你使用的任何类似服务）

如果您想深入了解 OpenRouter、模型选择或其他方面，请随时提问！